# The appendix tables, rebuilt from the modules that define them

Six tables the thesis prints that no script produced. Each was typed by hand
from a module, and a module can change under a typed table without anything
noticing.

| Table | Reports | Authority |
|---|---|---|
| 3.1 | The capability fields distinguishing the two arm types | `core/cell/cell_config.py` `ARM_TYPES`, `ARMS` |
| 3.2 | The five constraints the validator checks | `harvest/probe_store.py` |
| A.1 | Both object casts | `ycb/ycb_objects.py`, `ycb/layouts.py`, plus the capability rule |
| A.2 | The three name-swap pairs | `experiments/ex1/mislabel.py` `SWAP_PAIRS` |
| C.1 | The Experiment 2 field aliases | `experiments/ex2/prompts.py` `FIELD_ALIASES` |
| D.1 | Where the 162 frozen states came from | `probes/ex1_v2.json` |

Each section rebuilds its table, prints it, writes the LaTeX, and asserts the
numbers against what the thesis prints. A section that completes silently is a
table that still agrees with the write-up.

Checks read the thesis LaTeX from `THESIS_REPO`, falling back to
`thesis_expected.json` beside this notebook so a clone without the thesis tree
still checks. When both are present they must agree.

Tables F.1 and G.1 are generated by `notebooks/ex1/ex1_reproduce_tables.ipynb`,
which owns the Chapter 4 tables. **B.1, B.2 and E.1 to E.4 are still typed by
hand** and are the remaining work.

In [1]:
import collections, json, os, sys

# The package root: the nearest directory at or above the working directory
# that holds both probes/ and experiments/. Found by walking up rather than
# from __file__, so the notebook works wherever it is opened from.
ROOT = None
cand = os.path.abspath(os.getcwd())
while True:
    if (os.path.isdir(os.path.join(cand, "probes"))
            and os.path.isdir(os.path.join(cand, "experiments"))):
        ROOT = cand
        break
    parent = os.path.dirname(cand)
    if parent == cand:
        raise SystemExit("run this from fourarm/ or below")
    cand = parent
for p in (ROOT, os.path.join(ROOT, "ycb")):
    if p not in sys.path:
        sys.path.insert(0, p)

import thesis_check as TC
from core.cell import cell_config as C
from ycb_objects import YCB
from layouts import LAYOUT_CASTS
from experiments.ex1.mislabel import SWAP_PAIRS, CONTROLS, APERTURE_FRANKA
from experiments.ex2 import prompts as P2

TEX_OUT = os.path.join(ROOT, "tables", "appendix")
os.makedirs(TEX_OUT, exist_ok=True)
EXPECTED = os.path.join(ROOT, "notebooks", "appendix", "thesis_expected.json")
CK = TC.Checker(EXPECTED)

print("root        :", ROOT)
print("thesis repo :", CK.repo, "" if CK.have_thesis() else "  (not found)")
print("vendored    :", len(CK.expected), "numeric +", len(CK.expected_bodies),
      "body tables")


def show(header, rows):
    """Print a table without needing pandas."""
    cols = [max(len(str(h)), *(len(str(r[i])) for r in rows)) if rows
            else len(str(h)) for i, h in enumerate(header)]
    line = "  ".join("-" * c for c in cols)
    print("  ".join(str(h).ljust(c) for h, c in zip(header, cols)))
    print(line)
    for r in rows:
        print("  ".join(str(c).ljust(w) for c, w in zip(r, cols)))
    print(line)


def latex(label, caption, colspec, header, body_lines, note=None):
    """A table in the thesis's own shape, so the token sequences line up."""
    out = [r"\begin{table}[H]", r"\centering",
           r"\caption{%s}" % caption, r"\label{%s}" % label,
           r"\begin{tabular}{%s}" % colspec, r"\toprule",
           header + r" \\", r"\midrule"] + body_lines + [
           r"\bottomrule", r"\end{tabular}"]
    if note:
        out.append(r"\begin{tablenotes}\footnotesize\item %s\end{tablenotes}" % note)
    out += [r"\end{table}", ""]
    return "\n".join(out)


def write_tex(name, tex):
    path = os.path.join(TEX_OUT, name)
    with open(path, "w") as fh:
        fh.write(tex)
    print("wrote %s" % os.path.relpath(path, ROOT))
    return path


def esc(s):
    return str(s).replace("_", r"\_")

root        : /Users/erinsarlak/Downloads/MastersDissertation/fourarm
thesis repo : /Users/erinsarlak/Desktop/msc-paper 
vendored    : 0 numeric + 0 body tables


---
# 1. Table 3.1 — the capability fields that distinguish the two arm types

Reach, aperture, payload and the delicate flag, with how many arms of each type
the cell has. The aperture difference is nearly a factor of two, which is why
grasp is the constraint both experiments manipulate: it binds often enough on
this object set to be measurable.

In [2]:
counts = collections.Counter(a["type"] for a in C.ARMS.values())
ORDER = ["ur10", "franka"]          # the order the thesis prints them in
PRINT = {"ur10": "UR10", "franka": "Franka"}

rows_31 = []
for t in ORDER:
    s = C.ARM_TYPES[t]
    rows_31.append([PRINT[t], counts[t], "%.3f" % s["reach"],
                    "%.3f" % s["max_grasp_m"], "%g" % s["payload_kg"],
                    "yes" if s["delicate_ok"] else "no"])

show(["Arm type", "Count", "Reach (m)", "Aperture (m)", "Payload (kg)",
      "Delicate objects"], rows_31)

tex_31 = latex(
    "tab:system:arms",
    "The capability fields that distinguish the two arm types.",
    "lccccc",
    r"Arm type & Count & Reach ($\mathrm{m}$) & Aperture ($\mathrm{m}$) & "
    r"Payload ($\mathrm{kg}$) & Delicate objects",
    ["%s \\\\" % " & ".join(str(c) for c in r) for r in rows_31])
write_tex("system_arms.tex", tex_31)
CK.check("tab:system:arms", tex_31)

Arm type  Count  Reach (m)  Aperture (m)  Payload (kg)  Delicate objects
--------  -----  ---------  ------------  ------------  ----------------
UR10      2      1.300      0.140         10            no              
Franka    2      0.855      0.080         3             yes             
--------  -----  ---------  ------------  ------------  ----------------
wrote tables/appendix/system_arms.tex
tab:system:arms          MATCHES the thesis on all 9 numeric cells


True

---
# 2. Table 3.2 — the five constraints the validator checks

The validator returns one binding cause per rejected pair. Capability is
decomposed into its three sub-limits — grasp, payload, delicacy — because the
cell reports all three under a single code, and Experiment 1 needs to know
which of them bound.

This table carries no numbers, so it is checked on the constraint names rather
than on a token sequence.

In [3]:
# The vocabulary is the validator's own: capability_cause returns the three
# sub-limits and the reason-string branches supply the other two.
CAUSES = [
    ("Reach", "reach",
     "The arm can reach both the object's current position and the task's "
     "destination zone."),
    ("Grasp", "grasp",
     "The object's grasp width in its current pose does not exceed the arm's "
     "gripper aperture."),
    ("Delicacy", "delicate",
     "An object flagged delicate is assigned only to an arm cleared for "
     "delicate handling."),
    ("Payload", "payload",
     "The object's mass does not exceed the arm's payload limit."),
    ("Route", "no_route",
     "The path the task requires is available to that arm, accounting for "
     "zone occupancy."),
]

show(["Constraint", "Validator cause", "The check"],
     [[n, c, d[:56] + ("..." if len(d) > 56 else "")] for n, c, d in CAUSES])

tex_32 = latex(
    "tab:system:constraints",
    "The five constraints the validator checks.", "ll",
    "Constraint & The check",
    ["%s & %s \\\\" % (n, d) for n, _, d in CAUSES])
write_tex("system_constraints.tex", tex_32)

body, origin = CK.body("tab:system:constraints")
if body is None:
    CK.note("tab:system:constraints", "SKIPPED", "no thesis source, no vendored copy")
    print("tab:system:constraints   SKIPPED")
else:
    missing = [n for n, _, _ in CAUSES if n not in body]
    assert not missing, "constraint names missing from the thesis: %s" % missing
    CK.note("tab:system:constraints", "MATCHES",
            "five constraint names against the %s, no numeric cells" % origin)
    print("tab:system:constraints   MATCHES the %s on all %d constraint names"
          % (origin, len(CAUSES)))

Constraint  Validator cause  The check                                                  
----------  ---------------  -----------------------------------------------------------
Reach       reach            The arm can reach both the object's current position and...
Grasp       grasp            The object's grasp width in its current pose does not ex...
Delicacy    delicate         An object flagged delicate is assigned only to an arm cl...
Payload     payload          The object's mass does not exceed the arm's payload limi...
Route       no_route         The path the task requires is available to that arm, acc...
----------  ---------------  -----------------------------------------------------------
wrote tables/appendix/system_constraints.tex
tab:system:constraints   MATCHES the thesis on all 5 constraint names


---
# 3. Table A.1 — the two object casts

Cast A is the registry order the runner spawns; cast B is the disjoint set
named in `layouts.LAYOUT_CASTS`. The **Arms** column is not stored anywhere: it
is derived here by applying the capability rule to each object, so it cannot
disagree with the validator.

Two object names carry digits — `mug2` and `screw_99` — and those digits enter
the LaTeX token sequence. That is harmless as long as the generated table lists
the same objects in the same order, which is the point of generating it.

In [4]:
AP = {t: s["max_grasp_m"] for t, s in C.ARM_TYPES.items()}


def arms_for(o):
    """Which arm types satisfy the capability check for this object.

    Grasp span and the delicate flag both bind here; payload never does on
    either cast, the heaviest object being the wood block at 1.580 kg against
    the Franka limit of 3 kg.
    """
    ok = [t for t, ap in AP.items()
          if o["grasp_m"] <= ap
          and (not o["delicate"] or C.ARM_TYPES[t]["delicate_ok"])]
    if set(ok) == {"ur10", "franka"}:
        return "Both"
    if ok == ["franka"]:
        return "Franka only"
    if ok == ["ur10"]:
        return "UR only"
    return "none"


CAST_A = list(YCB)[:11]
CAST_B = LAYOUT_CASTS["set_b"]
assert not set(CAST_A) & set(CAST_B), "the casts must be disjoint"

rows_a1, body_a1 = [], []
for cast, names in (("Cast A", CAST_A), ("Cast B", CAST_B)):
    body_a1.append(r"\multicolumn{6}{l}{\emph{%s}} \\" % cast)
    for n in names:
        o = YCB[n]
        rows_a1.append([cast, n, o["category"], "%.3f" % o["grasp_m"],
                        "%.3f" % o["mass_kg"],
                        "yes" if o["delicate"] else "-", arms_for(o)])
        body_a1.append(r"%s & %s & %.3f & %.3f & %s & %s \\"
                       % (esc(n), o["category"], o["grasp_m"], o["mass_kg"],
                          r"\checkmark" if o["delicate"] else "--", arms_for(o)))

show(["Cast", "Object", "Category", "Grasp width (m)", "Mass (kg)",
      "Delicate", "Arms"], rows_a1)

tex_a1 = latex(
    "tab:appendix:cast",
    "The two object casts.", "llcccl",
    r"Object & Category & Grasp width (m) & Mass (kg) & Delicate & Arms",
    body_a1)
write_tex("appendix_cast.tex", tex_a1)
CK.check("tab:appendix:cast", tex_a1)

n_both = sum(1 for r in rows_a1 if r[0] == "Cast A" and r[6] == "Both")
print()
print("cast A: %d objects, %d graspable by both types; cast B: %d objects"
      % (len(CAST_A), n_both, len(CAST_B)))

Cast    Object         Category     Grasp width (m)  Mass (kg)  Delicate  Arms       
------  -------------  -----------  ---------------  ---------  --------  -----------
Cast A  soup_can       food         0.068            0.349      -         Both       
Cast A  banana         food         0.039            0.236      yes       Franka only
Cast A  gelatin_box    food         0.073            0.176      -         Both       
Cast A  meat_can       food         0.084            0.407      -         UR only    
Cast A  mustard        food         0.096            0.603      -         UR only    
Cast A  mug            kitchenware  0.081            0.493      -         UR only    
Cast A  mug2           kitchenware  0.081            0.493      -         UR only    
Cast A  bowl           kitchenware  0.030            0.670      yes       Franka only
Cast A  power_drill    tools        0.050            1.216      -         Both       
Cast A  large_clamp    tools        0.122            0

---
# 4. Table A.2 — the three name-swap pairs

A pair qualifies only when all three hold at once: the two objects lie on
opposite sides of the Franka aperture, neither is delicate, and they share a
category. The module that performed the swap is the authority, so the table is
built from it rather than restated.

In [5]:
# mislabel.py names objects as the STATE does, with a ycb_ prefix; the
# registry keys them without one. The prefix is stripped rather than the two
# being reconciled, because each is right for where it is used.
def reg(name):
    return name[4:] if name.startswith("ycb_") else name


rows_a2, body_a2 = [], []
for hi_id, lo_id in SWAP_PAIRS:
    hi, lo = reg(hi_id), reg(lo_id)
    o_hi, o_lo = YCB[hi], YCB[lo]
    assert o_hi["category"] == o_lo["category"], "a swap pair must share a category"
    assert not o_hi["delicate"] and not o_lo["delicate"], "neither may be delicate"
    assert o_hi["grasp_m"] > APERTURE_FRANKA >= o_lo["grasp_m"], \
        "a pair must straddle the Franka aperture"
    rows_a2.append([o_hi["category"], hi, "%.3f" % o_hi["grasp_m"],
                    lo, "%.3f" % o_lo["grasp_m"]])
    body_a2.append(r"%s & %s & %.3f & %s & %.3f \\"
                   % (o_hi["category"], esc(hi), o_hi["grasp_m"],
                      esc(lo), o_lo["grasp_m"]))

show(["Category", "Above the aperture", "Grasp width (m)",
      "Below the aperture", "Grasp width (m)"], rows_a2)

tex_a2 = latex(
    "tab:appendix:swappairs",
    "The three name swap pairs on cast A.", "lcccc",
    r"Category & Object & Grasp width (m) & Object & Grasp width (m)",
    body_a2)
write_tex("appendix_swappairs.tex", tex_a2)
CK.check("tab:appendix:swappairs", tex_a2)

print()
print("unswapped in-prompt controls: %s"
      % ", ".join(reg(c) for c in CONTROLS))
print("Franka aperture: %.3f m" % APERTURE_FRANKA)

Category  Above the aperture  Grasp width (m)  Below the aperture  Grasp width (m)
--------  ------------------  ---------------  ------------------  ---------------
tools     large_clamp         0.122            power_drill         0.050          
food      mustard             0.096            soup_can            0.068          
food      meat_can            0.084            gelatin_box         0.073          
--------  ------------------  ---------------  ------------------  ---------------
wrote tables/appendix/appendix_swappairs.tex
tab:appendix:swappairs   MATCHES the thesis on all 6 numeric cells

unswapped in-prompt controls: mug, mug2, banana, bowl
Franka aperture: 0.080 m


---
# 5. Table C.1 — the Experiment 2 field aliases

The state is rendered through an alias table, so the model never sees the name
the simulator uses. Two properties of the aliases are load-bearing: each
constraint becomes a visibly matched pair, so the direction of the comparison
is readable from the names alone, and `pose` becomes `resting_face` because a
posture label could be read off as an outcome where a geometric name cannot.

No numbers, so this is checked on the alias names.

In [6]:
rows_c1 = [[k, v] for k, v in P2.FIELD_ALIASES.items()]
show(["Internal name", "Name the model sees"], rows_c1)

tex_c1 = latex(
    "tab:appC:aliases",
    "The field aliases. The model never sees the name in the left column.",
    "ll", r"Internal name & Name the model sees",
    [r"\texttt{%s} & \texttt{%s} \\" % (esc(k), esc(v)) for k, v in rows_c1])
write_tex("appC_aliases.tex", tex_c1)

body, origin = CK.body("tab:appC:aliases")
if body is None:
    CK.note("tab:appC:aliases", "SKIPPED", "no thesis source, no vendored copy")
    print("tab:appC:aliases         SKIPPED")
else:
    # Matched on the alias the model SEES, which is the column that would
    # change the experiment if it drifted.
    missing = [v for _, v in rows_c1 if v.replace("_", r"\_") not in body
               and v not in body]
    assert not missing, "aliases missing from the thesis: %s" % missing
    CK.note("tab:appC:aliases", "MATCHES",
            "%d alias names against the %s, no numeric cells"
            % (len(rows_c1), origin))
    print("tab:appC:aliases         MATCHES the %s on all %d alias names"
          % (origin, len(rows_c1)))

Internal name  Name the model sees
-------------  -------------------
grasp_m        opening_needed_m   
max_grasp_m    opening_max_m      
payload_kg     max_load_kg        
delicate_ok    handles_delicate   
dims_m         size_upright_m     
reach_ok_arms  arms_that_can_reach
pose           resting_face       
-------------  -------------------
wrote tables/appendix/appC_aliases.tex
tab:appC:aliases         MATCHES the thesis on all 7 alias names


---
# 6. Table D.1 — where the 162 frozen states came from

Counted from the probe set itself rather than from a record of the harvest, so
the table describes the file the experiments actually read. The allocator column
matters: no run used a VLM, so the set is not shaped by any model's behaviour.

In [7]:
probes = json.load(open(os.path.join(ROOT, "probes", "ex1_v2.json")))["probes"]
by_source = collections.Counter(p["provenance"]["source"] for p in probes)

# The allocator each harvest ran under. rec_ is a recording of the rule
# allocator; every rnd_ run is random-valid.
def allocator(source):
    return "rule" if source.startswith("rec_") else "random-valid"


rows_d1 = [[s, n, allocator(s)] for s, n in by_source.most_common()]
show(["Source", "States", "Allocator"], rows_d1 + [["Total", sum(by_source.values()), ""]])

tex_d1 = latex(
    "tab:appendix:sources",
    "Sources of the %d states in \\texttt{ex1\\_v2}." % len(probes),
    "lrl", r"Source & States & Allocator",
    [r"\texttt{%s} & %d & %s \\" % (esc(s), n, allocator(s))
     for s, n in by_source.most_common()]
    + [r"\midrule", r"\textbf{Total} & \textbf{%d} & \\" % len(probes)])
write_tex("appendix_sources.tex", tex_d1)
CK.check("tab:appendix:sources", tex_d1)

assert not any(a == "vlm" for a in (allocator(s) for s in by_source)), \
    "a VLM-driven harvest would shape the probe set by a model's behaviour"
print()
print("%d states from %d runs, none harvested under a VLM"
      % (len(probes), len(by_source)))

Source             States  Allocator   
-----------------  ------  ------------
rnd_contention     34      random-valid
rnd_balanced       33      random-valid
rnd_relay          31      random-valid
rnd_captrap        31      random-valid
rnd_decision_rich  20      random-valid
rec_decision_rich  13      rule        
Total              162                 
-----------------  ------  ------------
wrote tables/appendix/appendix_sources.tex
tab:appendix:sources     MATCHES the thesis on all 7 numeric cells

162 states from 6 runs, none harvested under a VLM


---
# 7. Audit

One row per table: what it reports, the module that is the authority for it,
and whether the generated table still matches the thesis.

In [8]:
SPEC = [
    ("3.1", "tab:system:arms", "1", "arm capability fields",
     "core/cell/cell_config.py"),
    ("3.2", "tab:system:constraints", "2", "the five validator constraints",
     "harvest/probe_store.py"),
    ("A.1", "tab:appendix:cast", "3", "both object casts",
     "ycb/ycb_objects.py + layouts.py + the capability rule"),
    ("A.2", "tab:appendix:swappairs", "4", "the three name-swap pairs",
     "experiments/ex1/mislabel.py"),
    ("C.1", "tab:appC:aliases", "5", "the Experiment 2 field aliases",
     "experiments/ex2/prompts.py"),
    ("D.1", "tab:appendix:sources", "6", "where the 162 states came from",
     "probes/ex1_v2.json"),
]
rows = CK.audit_rows(SPEC)
show(["Table", "Label", "Built in", "Reports", "Authority", "Against thesis"], rows)

matched = sum(1 for r in rows if r[-1] == "MATCHES")
print()
print("%d of %d appendix tables regenerated and matched against the thesis"
      % (matched, len(rows)))
failed = [r for r in rows if r[-1] not in ("MATCHES", "SKIPPED")]
assert not failed, "tables not verified: %s" % [r[1] for r in failed]

Table  Label                   Built in  Reports                         Authority                                              Against thesis
-----  ----------------------  --------  ------------------------------  -----------------------------------------------------  --------------
3.1    tab:system:arms         §1        arm capability fields           core/cell/cell_config.py                               MATCHES       
3.2    tab:system:constraints  §2        the five validator constraints  harvest/probe_store.py                                 MATCHES       
A.1    tab:appendix:cast       §3        both object casts               ycb/ycb_objects.py + layouts.py + the capability rule  MATCHES       
A.2    tab:appendix:swappairs  §4        the three name-swap pairs       experiments/ex1/mislabel.py                            MATCHES       
C.1    tab:appC:aliases        §5        the Experiment 2 field aliases  experiments/ex2/prompts.py                             MATCHES       

---
# 8. Refreshing the vendored expectations

Written only when the thesis itself was readable. A run that fell back to the
vendored copy cannot refresh it — that would be the file certifying itself.

In [9]:
print(CK.refresh())

wrote thesis_expected.json: 4 numeric tables + 2 body tables, 66 numbers  (CHANGED - commit it)
